# R4-U1b (1/2): AST-A warm-up under Protocol A

Select a Colab **T4 GPU** runtime. Sync the new code, `configs/r4_u1_ast_warmup_a.json`, `configs/r4_u1_ast_ft_a.json`, the B recipes `configs/r2_u2_ast_b.json` and `configs/r3_u2_ast_ft_b.json`, the R4-S1 outputs, the frozen manifests and the cohort shards to Drive first. Run the cells in order.

This trains the AST classifier head only (the backbone stays frozen) on the Protocol A training crops with the B R2-U2 recipe: 2 epochs, one 4-second crop per recording per epoch, about 15 minutes on a T4. It writes `warmup_result.json`, the initializer and epoch-0 reference of the fine-tune (notebook 2/2). No test audio is read. The recipe's `status` must record the team's agreement or the run is refused.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
LEDGER = ROOT / 'reports/ledger.csv'
OWNER = 'Usman'  # Person running this notebook; written to the ledger.
MODEL_CACHE = '/content/hf_cache'  # The pinned AST is downloaded here and hash-verified; Colab-local, not synced.
SHARDS_DRIVE = ROOT / 'data/processed/cohort_v1'
SHARDS = Path('/content/shards')  # local copy of the shards: much faster to read than Drive
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
sys.path.insert(0, str(ROOT / 'src'))
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

# For a new run leave RUN_ID blank. To resume, enter the exact previous RUN_ID (code, environment and recipe must be unchanged).
RUN_ID = ''
RESUME = bool(RUN_ID)
if not RESUME:
    RUN_ID = 'r4_u1_ast_warmup_A_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
RUN_DIR = ROOT / 'runs/r4' / RUN_ID
print('Run:', RUN_ID, 'Resume:', RESUME, 'Run folder:', RUN_DIR)
assert (ROOT / 'scripts/r4_u1_ast.py').is_file(), 'Sync the updated project code to Drive'

In [ ]:
import torch, transformers, numpy
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__)
assert torch.cuda.is_available(), 'Choose a T4 GPU runtime'
assert transformers.__version__ == '5.22.1', f'the B AST runs used transformers 5.22.1; found {transformers.__version__}'
lock = json.loads((ROOT / 'locks/B_lock_gr3.json').read_text(encoding='utf-8'))
contracts_sha = sha256(ROOT / 'src/echoproof/contracts.py')
assert contracts_sha in json.dumps(lock), f'src/echoproof/contracts.py differs from the GR3 lock ({contracts_sha[:12]}); refresh it from git'

required = ['configs/r4_u1_ast_warmup_a.json', 'configs/r4_u1_ast_ft_a.json', 'configs/r2_u2_ast_b.json', 'configs/r3_u2_ast_ft_b.json',
            'data/manifests/cohort_v1.parquet', 'data/splits/protocol_A_v1.parquet', 'data/processed/chunks_v1.parquet',
            'data/features/r4_s1_A/r4_s1_manifest.json', 'reports/r1/split_report.json', 'locks/B_lock_gr3.json']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing inputs: {missing}'
assert SHARDS_DRIVE.is_dir(), 'Set SHARDS_DRIVE to the canonical shard directory'
recipe = json.loads((ROOT / 'configs/r4_u1_ast_warmup_a.json').read_text(encoding='utf-8'))
assert recipe['status'].startswith('team_agreed'), f"recipe status is {recipe['status']!r}: the team's agreement must be recorded first"
contract = RUN_DIR.parent / (RUN_DIR.name + '_contract.json')
assert contract.is_file() if RESUME else not RUN_DIR.exists() and not contract.exists(), 'Check RUN_ID and the resume setting'

## Fixture tests
Any failure or skip must be resolved before training. They use a tiny random AST and check the recipes against the B recipes, the guarded A data, the warm-up and the fine-tune proofs.

In [ ]:
tests = ['tests/test_r4_u1_ast.py', 'tests/test_r4_u0_wavlm_cache.py', 'tests/test_r3_u2_ast.py', 'tests/test_r1_u2_ast.py', 'tests/test_m1_trainer.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], text=True, capture_output=True)
print(check.stdout[-4000:], check.stderr[-2000:])
assert check.returncode == 0 and ' skipped' not in check.stdout, 'Resolve test failures or skips before training'

## Copy the shards to local disk
Reading audio from Drive is slow; the 15 shards (about 3.5 GB) are copied once to Colab's disk. Nothing on Drive changes.

In [ ]:
import shutil
SHARDS.mkdir(exist_ok=True)
t0 = time.time()
for shard in sorted(SHARDS_DRIVE.glob('shard_*.tar')):
    target = SHARDS / shard.name
    if not target.exists():
        shutil.copyfile(shard, target)
    assert target.stat().st_size == shard.stat().st_size, shard.name
print(len(list(SHARDS.glob('shard_*.tar'))), 'shards on local disk,', round(time.time() - t0), 's')

## Train the A warm-up
Head only, backbone proved unchanged at the start, at the end and in `best.pt`. A disconnected run is resumed by entering its `RUN_ID` above and rerunning from the top.

In [ ]:
cmd = [sys.executable, 'scripts/r4_u1_ast.py', 'warmup', '--root', str(ROOT), '--shard-dir', str(SHARDS), '--run-dir', str(RUN_DIR.relative_to(ROOT)),
       '--ledger', str(LEDGER.relative_to(ROOT)), '--owner', OWNER, '--model-cache', MODEL_CACHE]
if RESUME:
    cmd.append('--resume')
subprocess.run(cmd, check=True)

## Verify the result and write the report

In [ ]:
result = json.loads((RUN_DIR / 'warmup_result.json').read_text(encoding='utf-8'))
assert result['task'] == 'R4-U1b-warmup' and result['protocol'] == 'A' and result['problems'] == [] and result['epochs_run'] == 2
assert sha256(RUN_DIR / 'best.pt') == result['best_checkpoint_sha256']
pins = json.loads((ROOT / 'configs/r3_u2_ast_ft_b.json').read_text(encoding='utf-8'))['init']['pretrained_group_sha256']
assert all(result['group_sha256_best'][g] == h for g, h in pins.items()), 'a backbone group differs from the pinned pretrained weights'
with LEDGER.open(newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r['run_id'] == RUN_ID and r['status'] == 'completed']
assert len(rows) == 1, 'expected one completed ledger row'
print('best epoch', result['best_epoch'], '| selection macro F1', result['best_macro_f1'], '| skipped steps', result['skipped_steps'])
report = {'task': 'R4-U1b-warmup', 'run_id': RUN_ID, 'run_dir': str(RUN_DIR), 'warmup_result_sha256': sha256(RUN_DIR / 'warmup_result.json'),
          'best_epoch': result['best_epoch'], 'best_macro_f1': result['best_macro_f1'], 'checkpoint_sha256': result['best_checkpoint_sha256'],
          'selection_macro_f1_by_epoch': result['selection_macro_f1_by_epoch'], 'skipped_steps': result['skipped_steps'],
          'ledger_rows': len(rows), 'torch': torch.__version__, 'transformers': transformers.__version__, 'gpu': torch.cuda.get_device_name(0)}
path = ROOT / 'reports/r4' / ('r4_u1_ast_warmup_colab_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '.json')
assert not path.exists()
path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding='utf-8')
print('report:', path)